# AI Surveillance - Early Fire & Smoke Warning Detection Demo

This notebook demonstrates **Early Fire & Smoke Detection**. By loading a specialized community-trained YOLO model (`touati-kamel/yolov8s-forest-fire-detection`) directly from Hugging Face Hub using the Ultralytics library, we can detect fire, smoke, and industrial emissions, generating early alarms.

### Google Colab Instructions:
1. Click the "Runtime" menu option and select **"Change runtime type"**. Make sure **T4 GPU** is selected for optimal performance.
2. Run the cells sequentially to install dependencies, run inference, and visualize results.

## 1. Install Dependencies


In [ ]:
# Install dependencies
!pip install ultralytics opencv-python matplotlib numpy


## 2. Import Libraries


In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os
import random
import urllib.request
%matplotlib inline


## 3. Load Specialized Fire/Smoke Model


In [ ]:
print("Downloading Fire/Smoke Detection model weights from Hugging Face Hub...")
# Standard YOLO class doesn't support custom HF repo identifiers natively.
# We download the 'best.pt' weights manually to bypass loader errors and ensure a robust Colab experience.
fire_model_url = "https://huggingface.co/touati-kamel/yolov8s-forest-fire-detection/resolve/main/best.pt"
fire_model_path = "fire_smoke_detector.pt"
if not os.path.exists(fire_model_path):
    print("Downloading model weight file...")
    urllib.request.urlretrieve(fire_model_url, fire_model_path)
    print("Model weights download completed successfully!")
else:
    print("Model weights already present locally.")

print("Loading Forest Fire & Smoke Detection model...")
model = YOLO(fire_model_path)
print("Model loaded successfully!")
print("Detected Classes:", model.names)


## 4. Create a Procedural Simulated Fire and Smoke Test Video


In [ ]:
# To ensure the notebook works perfectly without needing to start a real fire,
# we will programmatically generate a realistic test video containing expanding fire and smoke.
width, height = 640, 480
fps = 24
total_duration_sec = 8
total_frames = fps * total_duration_sec

video_input_path = "fire_input.mp4"
video_output_path = "fire_output_raw.mp4"
video_final_path = "fire_output.mp4"

print("Generating simulated fire & smoke test video...")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_input_path, fourcc, fps, (width, height))

# Fire sparks and smoke particle state
particles = []

for f in range(total_frames):
    # Draw a simulated industrial forest setting background
    frame = np.zeros((height, width, 3), dtype=np.uint8)
    frame[:, :] = (30, 40, 25) # Dark olive forest background
    
    # Ground tree trunk rectangle mockups
    cv2.rectangle(frame, (100, 380), (120, 480), (20, 30, 40), -1)
    cv2.rectangle(frame, (400, 350), (430, 480), (20, 30, 40), -1)
    
    # 1. Simulate Fire Ignition and Growth (starting at frame 20)
    if f > 20:
        fire_cx, fire_cy = 110, 380
        fire_radius = int(min(60, (f - 20) * 0.7))
        
        # Multi-layered flame circles (Core white-hot, inner yellow, outer red/orange)
        cv2.circle(frame, (fire_cx, fire_cy), fire_radius + random.randint(-4, 4), (0, 0, 240), -1) # Red
        cv2.circle(frame, (fire_cx, fire_cy), int(fire_radius * 0.7) + random.randint(-3, 3), (0, 165, 255), -1) # Orange
        cv2.circle(frame, (fire_cx, fire_cy), int(fire_radius * 0.4) + random.randint(-2, 2), (0, 255, 255), -1) # Yellow
        
        # 2. Simulate Smoke Particles blowing upwards
        if f % 2 == 0:
            # Spawn new particle
            particles.append({
                'x': float(fire_cx + random.randint(-15, 15)),
                'y': float(fire_cy - 10),
                'r': float(random.randint(10, 25)),
                'alpha': 180.0
            })
            
    # Process smoke particles
    overlay = frame.copy()
    for p in list(particles):
        # Move upwards and expand
        p['y'] -= 2.8
        p['x'] += random.uniform(-1.0, 1.5)
        p['r'] += 0.4
        p['alpha'] -= 1.8
        
        if p['alpha'] <= 0 or p['y'] < 0:
            particles.remove(p)
            continue
            
        # Draw smoke cloud particle (semi-transparent gray/brown)
        cv2.circle(overlay, (int(p['x']), int(p['y'])), int(p['r']), (90, 95, 95), -1)
        
    cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
    out.write(frame)
    
out.release()
print("Simulated surveillance video saved successfully!")


## 5. Run Fire/Smoke Inference on Video


In [ ]:
cap = cv2.VideoCapture(video_input_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_output_path, fourcc, fps, (width, height))

print("Running early warning AI inference loop...")
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    frame_idx += 1
    annotated_frame = frame.copy()
    
    # Run forest fire / smoke detection
    results = model.predict(source=frame, conf=0.25, imgsz=320, verbose=False)
    
    alarm_triggered = False
    
    if results and len(results) > 0 and results[0].boxes is not None:
        boxes = results[0].boxes
        for box in boxes:
            class_id = int(box.cls[0].item())
            class_name = model.names[class_id]
            conf = float(box.conf[0].item())
            
            x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
            
            # Color alerts: Red for Fire, Dark Gray/Yellow for Smoke
            if class_name in ['fire', 'fire-smoke']:
                box_color = (0, 0, 255)
                alarm_triggered = True
            else:
                box_color = (128, 128, 128) # Smoke
                
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), box_color, 2)
            label = f"{class_name.upper()} ({int(conf*100)}%)"
            
            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
            cv2.rectangle(annotated_frame, (x1, y1 - text_h - 6), (x1 + text_w + 10, y1), box_color, -1)
            cv2.putText(annotated_frame, label, (x1 + 5, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
            
    # Header Warning Alarm Banner
    if alarm_triggered:
        cv2.rectangle(annotated_frame, (0, 0), (width, 35), (0, 0, 255), -1)
        cv2.putText(annotated_frame, "!!! WARNING: ACTIVE FLAME OR IGNITION DETECTED !!!", (width // 2 - 210, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
    out.write(annotated_frame)
    
cap.release()
out.release()
print("Early warning video loop completed!")


## 6. Convert Output Video to H.264


In [ ]:
print("Transcoding simulated results using FFmpeg...")
os.system(f"ffmpeg -y -i {video_output_path} -vcodec libx264 -f mp4 {video_final_path}")
print("Transcoding complete!")


## 7. Play Transcoded Video Inline


In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open(video_final_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<div style="text-align: center;">
    <h3 style="color: #ef4444; font-family: sans-serif;">Early Warning Surveillance Stream</h3>
    <video width="640" height="480" controls autoplay loop style="border: 2px solid #ef4444; border-radius: 12px; box-shadow: 0 5px 15px rgba(239, 68, 68, 0.4);">
        <source src="{data_url}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
</div>
""")


## 8. Setup Colab Webcam Bridge Utilities


In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
import PIL.Image

def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))
    return bbox_bytes


## 9. Initialize Web Camera Stream CSS/JS Component


In [ ]:
def start_webcam_stream():
    js_code = """
    var video;
    var container;
    var canvas;
    var labelVal;
    var audioCtx;
    
    function playAlarmSound() {
      try {
        if (!audioCtx) audioCtx = new (window.AudioContext || window.webkitAudioContext)();
        const osc = audioCtx.createOscillator();
        const gain = audioCtx.createGain();
        osc.type = 'sawtooth';
        osc.frequency.setValueAtTime(880, audioCtx.currentTime);
        gain.gain.setValueAtTime(0.08, audioCtx.currentTime);
        osc.connect(gain);
        gain.connect(audioCtx.destination);
        osc.start();
        osc.stop(audioCtx.currentTime + 0.15);
      } catch (e) {
        console.log('Audio error:', e);
      }
    }
    
    async function initWebcam() {
      container = document.createElement('div');
      container.style.display = 'flex';
      container.style.flexDirection = 'column';
      container.style.alignItems = 'center';
      container.style.fontFamily = 'system-ui, -apple-system, sans-serif';
      container.style.background = '#110505';
      container.style.padding = '20px';
      container.style.borderRadius = '16px';
      container.style.boxShadow = '0 10px 35px rgba(239,68,68,0.2)';
      container.style.width = '680px';
      container.style.margin = '20px auto';
      container.style.color = '#ffffff';
      container.style.border = '2px solid #571c1c';
      container.style.transition = 'all 0.3s';
      
      const title = document.createElement('h3');
      title.innerText = 'Early Flame & Smoke Detection Telemetry';
      title.style.margin = '0 0 15px 0';
      title.style.color = '#ef4444';
      title.style.fontSize = '1.3rem';
      container.appendChild(title);
      
      const videoWrapper = document.createElement('div');
      videoWrapper.style.position = 'relative';
      videoWrapper.style.width = '640px';
      videoWrapper.style.height = '480px';
      videoWrapper.style.borderRadius = '10px';
      videoWrapper.style.overflow = 'hidden';
      videoWrapper.style.border = '2px solid #571c1c';
      
      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 640;
      video.height = 480;
      videoWrapper.appendChild(video);
      
      canvas = document.createElement('canvas');
      canvas.style.position = 'absolute';
      canvas.style.top = '0';
      canvas.style.left = '0';
      canvas.width = 640;
      canvas.height = 480;
      videoWrapper.appendChild(canvas);
      container.appendChild(videoWrapper);
      
      labelVal = document.createElement('div');
      labelVal.innerText = 'Sensors Calibrated. Monitoring for fire hazards...';
      labelVal.style.marginTop = '15px';
      labelVal.style.fontSize = '1rem';
      labelVal.style.fontWeight = 'bold';
      labelVal.style.color = '#10b981';
      container.appendChild(labelVal);
      
      const stopBtn = document.createElement('button');
      stopBtn.innerText = 'DISARM FIRE SENSORS';
      stopBtn.style.marginTop = '15px';
      stopBtn.style.padding = '12px 24px';
      stopBtn.style.background = '#3f3f46';
      stopBtn.style.color = 'white';
      stopBtn.style.border = 'none';
      stopBtn.style.borderRadius = '8px';
      stopBtn.style.cursor = 'pointer';
      stopBtn.style.fontWeight = 'bold';
      stopBtn.style.transition = 'all 0.2s';
      stopBtn.onmouseover = () => stopBtn.style.background = '#52525b';
      stopBtn.onmouseout = () => stopBtn.style.background = '#3f3f46';
      
      stopBtn.onclick = () => {
        window.webcamActive = false;
        labelVal.innerText = 'Sensors Offline. Disarmed.';
        labelVal.style.color = '#71717a';
        container.style.borderColor = '#27272a';
        container.style.background = '#18181b';
        const tracks = video.srcObject.getTracks();
        tracks.forEach(track => track.stop());
        video.srcObject = null;
        stopBtn.disabled = true;
        stopBtn.style.cursor = 'not-allowed';
      };
      container.appendChild(stopBtn);
      
      document.body.appendChild(container);
      
      const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 } });
      video.srcObject = stream;
      await video.play();
      
      window.webcamActive = true;
    }
    
    async function captureFrame() {
      if (!window.webcamActive) return null;
      const tempCanvas = document.createElement('canvas');
      tempCanvas.width = 640;
      tempCanvas.height = 480;
      tempCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
      return tempCanvas.toDataURL('image/jpeg', 0.8);
    }
    
    function drawOverlay(base64Png, isAlarm) {
      if (!window.webcamActive) return;
      const ctx = canvas.getContext('2d');
      ctx.clearRect(0, 0, 640, 480);
      
      if (base64Png) {
        const img = new Image();
        img.onload = function() {
          ctx.drawImage(img, 0, 0);
        };
        img.src = base64Png;
      }
      
      if (isAlarm) {
        labelVal.innerText = '!!! WARNING: FIRE HAZARD DETECTED !!!';
        labelVal.style.color = '#ef4444';
        container.style.background = '#320000';
        container.style.borderColor = '#ef4444';
        container.style.boxShadow = '0 0 25px rgba(239, 68, 68, 0.6)';
        playAlarmSound();
      } else {
        labelVal.innerText = 'Sensors Calibrated. Monitoring for fire hazards...';
        labelVal.style.color = '#10b981';
        container.style.background = '#110505';
        container.style.borderColor = '#571c1c';
        container.style.boxShadow = '0 10px 35px rgba(239,68,68,0.2)';
      }
    }
    """
    display(Javascript(js_code))


## 10. Run Real-Time Webcam Fire/Smoke Stream Loop


In [ ]:
start_webcam_stream()
eval_js('initWebcam()')

try:
    while True:
        js_reply = eval_js('captureFrame()')
        if js_reply is None: break
        
        frame = js_to_image(js_reply)
        results = model.predict(source=frame, conf=0.25, imgsz=320, verbose=False)
        
        overlay_array = np.zeros((480, 640, 4), dtype=np.uint8)
        alarm_active = False
        
        if results and len(results) > 0 and results[0].boxes is not None:
            boxes = results[0].boxes
            for box in boxes:
                class_id = int(box.cls[0].item())
                class_name = model.names[class_id]
                conf = float(box.conf[0].item())
                
                x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                
                if class_name in ['fire', 'fire-smoke']:
                    box_color = (0, 0, 255, 255) # Red
                    alarm_active = True
                else:
                    box_color = (128, 128, 128, 255) # Gray
                    
                cv2.rectangle(overlay_array, (x1, y1), (x2, y2), box_color, 2)
                label = f"{class_name.upper()} ({int(conf*100)}%)"
                
                (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
                cv2.rectangle(overlay_array, (x1, y1 - text_h - 6), (x1 + text_w + 10, y1), box_color, -1)
                cv2.putText(overlay_array, label, (x1 + 5, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255, 255), 1)
                
        if alarm_active:
            cv2.rectangle(overlay_array, (0, 0), (640, 35), (0, 0, 255, 255), -1)
            cv2.putText(overlay_array, "!!! WARNING: ACTIVE FLAME OR IGNITION DETECTED !!!", (100, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255, 255), 2)
            
        bbox_bytes = bbox_to_bytes(overlay_array)
        eval_js(f'drawOverlay("{bbox_bytes}", {str(alarm_active).lower()})')
except Exception as e:
    print(f"Stream stopped: {e}")
